# Numerical Stability & Floating-Point Precision

## Definition
**Floating-point numbers** in computers have limited precision. This leads to rounding errors
that can accumulate and cause incorrect results — a major concern in AI/ML.

### IEEE 754 Floating-Point
| Type | Bits | Significant Digits | Epsilon |
|------|------|-------------------|---------|
| float16 | 16 | ~3.3 | 9.77e-4 |
| float32 | 32 | ~7.2 | 1.19e-7 |
| float64 | 64 | ~15.9 | 2.22e-16 |

### Key Concepts
- **Machine epsilon (eps)**: smallest number such that `1.0 + eps ≠ 1.0`
- **Underflow**: number too small to represent → becomes 0
- **Overflow**: number too large → becomes inf
- **Catastrophic cancellation**: subtracting nearly equal numbers loses precision
- **Numerical stability**: algorithms that don't amplify rounding errors

In [ ]:
import numpy as np

# Machine epsilon
for dtype in [np.float16, np.float32, np.float64]:
    info = np.finfo(dtype)
    print(f'{str(dtype):10s}  eps={info.eps:.3e}  min={info.min:.3e}  max={info.max:.3e}')

In [ ]:
# Floating-point arithmetic is NOT always exact
print('0.1 + 0.2 =', 0.1 + 0.2)           # 0.30000000000000004
print('0.1 + 0.2 == 0.3:', 0.1 + 0.2 == 0.3)  # False!

# Correct comparison: use np.isclose or np.allclose
a = np.float64(0.1 + 0.2)
b = np.float64(0.3)
print('np.isclose:', np.isclose(a, b))  # True
print('np.allclose([a], [b]):', np.allclose([a], [b]))

In [ ]:
# Catastrophic cancellation
a = np.float32(1_000_000.0)
b = np.float32(1_000_000.1)

diff = b - a   # should be 0.1, but float32 loses precision
print(f'float32: {b} - {a} = {diff}')  # may be 0.0 or imprecise!

# float64 is better
a64 = np.float64(1_000_000.0)
b64 = np.float64(1_000_000.1)
print(f'float64: {b64} - {a64} = {b64 - a64}')

In [ ]:
# Numerically stable log-sum-exp
def naive_log_sum_exp(x):
    return np.log(np.sum(np.exp(x)))

def stable_log_sum_exp(x):
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

x = np.array([1000.0, 1001.0, 1002.0])

print('Naive:', naive_log_sum_exp(x))    # inf (overflow!)
print('Stable:', stable_log_sum_exp(x)) # correct: ~1002.41

import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    x_neg = np.array([-1000.0, -1001.0, -1002.0])
    print('Naive underflow:', naive_log_sum_exp(x_neg))  # -inf
    print('Stable underflow:', stable_log_sum_exp(x_neg)) # correct

In [ ]:
# Numerically stable softmax
def softmax_naive(x):
    return np.exp(x) / np.sum(np.exp(x))

def softmax_stable(x):
    x = x - np.max(x)   # subtract max for stability
    e = np.exp(x)
    return e / e.sum()

logits = np.array([1000.0, 1001.0, 1002.0])
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print('Naive softmax:', softmax_naive(logits))    # [nan, nan, nan]!
print('Stable softmax:', softmax_stable(logits))  # correct

In [ ]:
# Epsilon for numerical safety (avoid division by zero)
# In batch normalization, layer norm, etc.
epsilon = 1e-8

arr = np.array([0.0, 1.0, 4.0])
safe_sqrt = np.sqrt(arr + epsilon)
print('Safe sqrt:', safe_sqrt)  # avoids sqrt(0) issues

# Safe log
safe_log = np.log(arr + epsilon)
print('Safe log:', safe_log)  # avoids log(0) = -inf

In [ ]:
# Kahan summation — compensated sum for better precision
def kahan_sum(arr):
    total = 0.0
    c = 0.0  # compensation
    for x in arr:
        y = x - c
        t = total + y
        c = (t - total) - y
        total = t
    return total

# Test with many small numbers
arr = np.ones(10000, dtype=np.float32) * 0.1
naive = np.float32(0.0)
for x in arr:
    naive += x
print(f'Naive float32 sum: {naive}')
print(f'np.sum float32: {np.sum(arr)}')
print(f'Kahan sum: {kahan_sum(arr)}')
print(f'Expected: {0.1 * 10000}')

In [ ]:
# float16 precision issues in deep learning
a = np.array([1.0], dtype=np.float16)
b = np.array([0.001], dtype=np.float16)

print('float16: 1.0 + 0.001 =', (a + b)[0])  # may round to 1.0!
print('float32: 1.0 + 0.001 =', (np.float32(1.0) + np.float32(0.001)))

# Mixed precision: compute in float32, store in float16
result_f16 = np.float32(1.0) + np.float32(0.001)
print('Mixed precision:', result_f16, '-> stored as f16:', np.float16(result_f16))